In [57]:
import os
from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict, Annotated, List
import operator
from langgraph.prebuilt import ToolNode
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [58]:
load_dotenv()

True

In [59]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [60]:
class ParallelState(TypedDict):
    query: str
    overview: str
    latest_news: str
    information: Annotated[List[str], operator.add]
    final_output: str

In [61]:
def overview_node(state: ParallelState):
    query = state["query"]
    llm_response = llm.invoke(
        f"Provide a brief overview of the following topic: {query}"
    )
    information = [llm_response.content]
    return {"overview": llm_response, "information": information}

In [62]:
def news_node(state: ParallelState):
    query = state["query"]
    search = DuckDuckGoSearchRun()
    search_response = search.run(query)
    information = [search_response]
    return {"latest_news": search_response, "information": information}

In [63]:
def information_node(state: ParallelState):
    information = state["information"]
    llm_response = llm.invoke(f"Summarize the following information: {information}")
    return {"final_output": llm_response.content}

In [64]:
workflow = StateGraph(ParallelState)
workflow.add_node("overview_node", overview_node)
workflow.add_node("news_node", news_node)
workflow.add_node("information_node", information_node)
workflow.add_edge(START, "overview_node")
workflow.add_edge(START, "news_node")
workflow.add_edge("overview_node", "information_node")
workflow.add_edge("news_node", "information_node")
workflow.add_edge("information_node", END)
app = workflow.compile()

In [65]:
app.invoke({"query": "What is the current state of AI research?"})

{'query': 'What is the current state of AI research?',
 'overview': AIMessage(content='**Current State of AI Research (2024‑2025)**  \n\n| Area | Key Developments | Open Challenges |\n|------|------------------|-----------------|\n| **Foundation & Large‑Scale Models** | • 2023‑24 saw a wave of “foundation models” with 10\u202fB–1\u202fT parameters, covering language, vision, audio, and multimodal tasks (e.g., GPT‑4‑Turbo, Gemini‑Pro, LLaMA‑3, Stable Diffusion‑XL). <br>• Prompt‑engineering, instruction‑tuning, and reinforcement‑learning‑from‑human‑feedback (RLHF) have become standard for aligning these models with user intent. | • Scaling laws plateau for many tasks; diminishing returns vs. compute cost. <br>• Data quality, bias, and privacy concerns remain. |\n| **Multimodal & Embodied AI** | • Unified models that process text, images, video, audio, and even sensor data (e.g., DeepMind’s Gato‑2, Meta’s Flamingo‑2). <br>• Progress in embodied agents that learn via simulation‑to‑real tra

In [ ]:
{
    "query": "What is the current state of AI research?",
    "overview": AIMessage(
        content="**Current State of AI Research (2024‑2025)**  \n\n| Area | Key Developments | Open Challenges |\n|------|------------------|-----------------|\n| **Foundation & Large‑Scale Models** | • 2023‑24 saw a wave of “foundation models” with 10\u202fB–1\u202fT parameters, covering language, vision, audio, and multimodal tasks (e.g., GPT‑4‑Turbo, Gemini‑Pro, LLaMA‑3, Stable Diffusion‑XL). <br>• Prompt‑engineering, instruction‑tuning, and reinforcement‑learning‑from‑human‑feedback (RLHF) have become standard for aligning these models with user intent. | • Scaling laws plateau for many tasks; diminishing returns vs. compute cost. <br>• Data quality, bias, and privacy concerns remain. |\n| **Multimodal & Embodied AI** | • Unified models that process text, images, video, audio, and even sensor data (e.g., DeepMind’s Gato‑2, Meta’s Flamingo‑2). <br>• Progress in embodied agents that learn via simulation‑to‑real transfer (e.g., robotic manipulation with MuJoCo‑based curricula). | • Robust perception‑action loops in the wild; handling out‑of‑distribution environments. |\n| **Efficient & Green AI** | • Sparse‑mixing, MoE (Mixture‑of‑Experts), quantization (4‑bit/2‑bit), and distillation pipelines cut inference cost by 5‑10×. <br>• Specialized hardware (TPU‑v5p, NVIDIA Hopper, Graphcore IPU) optimized for transformer workloads. | • Balancing efficiency with model fidelity; developing universal benchmarks for energy‑aware AI. |\n| **Neurosymbolic & Structured Reasoning** | • Hybrid systems that combine neural nets with symbolic reasoning (e.g., DeepMind’s AlphaTensor, IBM’s Neuro‑Symbolic AI). <br>• Advances in program synthesis, theorem proving, and causal inference. | • Seamless integration of discrete logic with differentiable components; interpretability of hybrid pipelines. |\n| **Reinforcement Learning (RL) & Decision‑Making** | • Offline RL, hierarchical RL, and diffusion‑based policy generation are enabling sample‑efficient learning. <br>• Successes in complex games (StarCraft\u202fII, Atari 100k) and real‑world domains (autonomous driving stacks, logistics). | • Safety in exploration, reward hacking, and transfer to non‑simulated environments. |\n| **AI Alignment & Safety** | • Growing research on interpretability, verification, and “steerability” of large models. <br>• Institutional efforts (OpenAI’s Safety team, DeepMind’s Alignment Lab, Partnership on AI) producing standards and red‑team evaluations. | • Scalable oversight, value learning, and mitigation of emergent deceptive behavior. |\n| **Ethics, Governance & Regulation** | • EU AI Act (2024) and U.S. AI Bill of Rights drafts shape compliance requirements (risk categorization, transparency, data provenance). <br>• Increased focus on bias audits, provenance tracking, and “model cards.” | • Harmonizing global regulatory regimes; enforcing accountability for black‑box systems. |\n| **Applications & Industry Adoption** | • Enterprise AI: generative code assistants, synthetic data generation, and AI‑augmented analytics. <br>• Healthcare: protein‑folding extensions, multimodal diagnostics, and drug‑discovery pipelines. <br>• Creative media: text‑to‑video, AI‑driven game design, and real‑time virtual avatars. | • Integration with legacy systems, data security, and domain‑specific validation. |\n\n### Take‑away Summary\n- **Momentum:** AI research continues to be driven by ever larger, more versatile foundation models and the push toward multimodal, embodied intelligence.\n- **Shift to Efficiency:** Because raw scaling is costly, the community emphasizes sparsity, quantization, and specialized hardware to make models greener and more deployable.\n- **Safety & Governance:** Parallel to technical advances, alignment, interpretability, and regulatory frameworks are gaining urgency, aiming to keep powerful models trustworthy and socially beneficial.\n- **Application Breadth:** From scientific discovery to everyday productivity tools, AI is moving from prototype to production across many sectors, but robust validation and responsible deployment remain critical bottlenecks.\n\nOverall, the field is at a crossroads where **capability** (massive, generalist models) meets **responsibility** (efficiency, safety, and governance), shaping the next wave of research and real‑world impact.",
        additional_kwargs={
            "reasoning_content": 'The user asks: "Provide a brief overview of the following topic: What is the current state of AI research?" They want a brief overview. Should summarize major trends, areas, breakthroughs, challenges, etc. Should be concise but cover key points: deep learning, foundation models, multimodal models, reinforcement learning, AI alignment, interpretability, hardware, regulation, applications, etc. Provide maybe bullet points. No disallowed content. It\'s fine.'
        },
        response_metadata={
            "token_usage": {
                "completion_tokens": 1061,
                "prompt_tokens": 89,
                "total_tokens": 1150,
                "completion_time": 2.245602648,
                "completion_tokens_details": {"reasoning_tokens": 91},
                "prompt_time": 0.003793307,
                "prompt_tokens_details": None,
                "queue_time": 0.042873268,
                "total_time": 2.249395955,
            },
            "model_name": "openai/gpt-oss-120b",
            "system_fingerprint": "fp_d81b3304b3",
            "service_tier": "on_demand",
            "finish_reason": "stop",
            "logprobs": None,
            "model_provider": "groq",
        },
        id="lc_run--019d67b2-dd97-7143-bc5a-c872febab866-0",
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            "input_tokens": 89,
            "output_tokens": 1061,
            "total_tokens": 1150,
            "output_token_details": {"reasoning": 91},
        },
    ),
    "latest_news": "Artificial general intelligence is a theoretical type of artificial intelligence that matches or surpasses human capabilities across virtually all cognitive tasks. TheStateofAIReportisthemost widely read and trusted analysis of key developments inAI. Published annually since 2018, the open-access report aims to spark informed conversation about thestateofAIand what it means for the future. ThestateofAIin 2025: Agents, innovation, and transformation.AIuse continues to broaden but remains primarily in pilot phases. Our latest survey shows a larger share of respondents reportingAIuse by their organizations, though most have yet to scale the technologies. AIwon't kill us all — but that doesn't make it trustworthy. Instead of getting distracted by future existential risks,AIethicsresearcherSasha Luccioni t... The site owner hides the web page description.",
    "information": [
        "Artificial general intelligence is a theoretical type of artificial intelligence that matches or surpasses human capabilities across virtually all cognitive tasks. TheStateofAIReportisthemost widely read and trusted analysis of key developments inAI. Published annually since 2018, the open-access report aims to spark informed conversation about thestateofAIand what it means for the future. ThestateofAIin 2025: Agents, innovation, and transformation.AIuse continues to broaden but remains primarily in pilot phases. Our latest survey shows a larger share of respondents reportingAIuse by their organizations, though most have yet to scale the technologies. AIwon't kill us all — but that doesn't make it trustworthy. Instead of getting distracted by future existential risks,AIethicsresearcherSasha Luccioni t... The site owner hides the web page description.",
        "**Current State of AI Research (2024‑2025)**  \n\n| Area | Key Developments | Open Challenges |\n|------|------------------|-----------------|\n| **Foundation & Large‑Scale Models** | • 2023‑24 saw a wave of “foundation models” with 10\u202fB–1\u202fT parameters, covering language, vision, audio, and multimodal tasks (e.g., GPT‑4‑Turbo, Gemini‑Pro, LLaMA‑3, Stable Diffusion‑XL). <br>• Prompt‑engineering, instruction‑tuning, and reinforcement‑learning‑from‑human‑feedback (RLHF) have become standard for aligning these models with user intent. | • Scaling laws plateau for many tasks; diminishing returns vs. compute cost. <br>• Data quality, bias, and privacy concerns remain. |\n| **Multimodal & Embodied AI** | • Unified models that process text, images, video, audio, and even sensor data (e.g., DeepMind’s Gato‑2, Meta’s Flamingo‑2). <br>• Progress in embodied agents that learn via simulation‑to‑real transfer (e.g., robotic manipulation with MuJoCo‑based curricula). | • Robust perception‑action loops in the wild; handling out‑of‑distribution environments. |\n| **Efficient & Green AI** | • Sparse‑mixing, MoE (Mixture‑of‑Experts), quantization (4‑bit/2‑bit), and distillation pipelines cut inference cost by 5‑10×. <br>• Specialized hardware (TPU‑v5p, NVIDIA Hopper, Graphcore IPU) optimized for transformer workloads. | • Balancing efficiency with model fidelity; developing universal benchmarks for energy‑aware AI. |\n| **Neurosymbolic & Structured Reasoning** | • Hybrid systems that combine neural nets with symbolic reasoning (e.g., DeepMind’s AlphaTensor, IBM’s Neuro‑Symbolic AI). <br>• Advances in program synthesis, theorem proving, and causal inference. | • Seamless integration of discrete logic with differentiable components; interpretability of hybrid pipelines. |\n| **Reinforcement Learning (RL) & Decision‑Making** | • Offline RL, hierarchical RL, and diffusion‑based policy generation are enabling sample‑efficient learning. <br>• Successes in complex games (StarCraft\u202fII, Atari 100k) and real‑world domains (autonomous driving stacks, logistics). | • Safety in exploration, reward hacking, and transfer to non‑simulated environments. |\n| **AI Alignment & Safety** | • Growing research on interpretability, verification, and “steerability” of large models. <br>• Institutional efforts (OpenAI’s Safety team, DeepMind’s Alignment Lab, Partnership on AI) producing standards and red‑team evaluations. | • Scalable oversight, value learning, and mitigation of emergent deceptive behavior. |\n| **Ethics, Governance & Regulation** | • EU AI Act (2024) and U.S. AI Bill of Rights drafts shape compliance requirements (risk categorization, transparency, data provenance). <br>• Increased focus on bias audits, provenance tracking, and “model cards.” | • Harmonizing global regulatory regimes; enforcing accountability for black‑box systems. |\n| **Applications & Industry Adoption** | • Enterprise AI: generative code assistants, synthetic data generation, and AI‑augmented analytics. <br>• Healthcare: protein‑folding extensions, multimodal diagnostics, and drug‑discovery pipelines. <br>• Creative media: text‑to‑video, AI‑driven game design, and real‑time virtual avatars. | • Integration with legacy systems, data security, and domain‑specific validation. |\n\n### Take‑away Summary\n- **Momentum:** AI research continues to be driven by ever larger, more versatile foundation models and the push toward multimodal, embodied intelligence.\n- **Shift to Efficiency:** Because raw scaling is costly, the community emphasizes sparsity, quantization, and specialized hardware to make models greener and more deployable.\n- **Safety & Governance:** Parallel to technical advances, alignment, interpretability, and regulatory frameworks are gaining urgency, aiming to keep powerful models trustworthy and socially beneficial.\n- **Application Breadth:** From scientific discovery to everyday productivity tools, AI is moving from prototype to production across many sectors, but robust validation and responsible deployment remain critical bottlenecks.\n\nOverall, the field is at a crossroads where **capability** (massive, generalist models) meets **responsibility** (efficiency, safety, and governance), shaping the next wave of research and real‑world impact.",
    ],
    "final_output": "**Brief Overview**\n\n- **Artificial General Intelligence (AGI)** – a theoretical AI that can match or exceed human performance on essentially any cognitive task.  \n- **The State of AI Report** – the most‑read, open‑access annual analysis of AI progress (published each year since 2018). The 2025 edition stresses that AI adoption is still largely experimental (pilot‑phase projects) and that scaling remains limited. It also argues that, while existential threats are often overstated, the real challenge is building trustworthy, ethically‑aligned systems.\n\n---\n\n### Current State of AI Research (2024‑2025)\n\n| Research Area | What’s New (Key Developments) | What Still Needs Work (Open Challenges) |\n|---------------|------------------------------|------------------------------------------|\n| **Foundation & Large‑Scale Models** | • “Foundation models” with 10\u202fB–1\u202fT parameters dominate (GPT‑4‑Turbo, Gemini‑Pro, LLaMA‑3, Stable Diffusion‑XL). <br>• Prompt‑engineering, instruction‑tuning, RLHF are standard for alignment. | • Scaling laws show diminishing returns vs. compute cost. <br>• Data quality, bias, privacy. |\n| **Multimodal & Embodied AI** | • Unified models handling text, images, video, audio, sensor data (e.g., Gato‑2, Flamingo‑2). <br>• Embodied agents learn in simulation and transfer to real robots. | • Robust perception‑action loops in uncontrolled environments; out‑of‑distribution handling. |\n| **Efficient & Green AI** | • Sparse‑mixing, Mixture‑of‑Experts, 4‑/2‑bit quantization, distillation cut inference cost 5‑10×. <br>• New hardware (TPU‑v5p, NVIDIA Hopper, Graphcore IPU) optimized for transformers. | • Trade‑off between efficiency and model fidelity; lack of universal energy‑aware benchmarks. |\n| **Neurosymbolic & Structured Reasoning** | • Hybrid neural‑symbolic systems (AlphaTensor, IBM Neuro‑Symbolic AI). <br>• Progress in program synthesis, theorem proving, causal inference. | • Seamless integration of discrete logic with differentiable parts; interpretability of hybrid pipelines. |\n| **Reinforcement Learning & Decision‑Making** | • Offline, hierarchical, and diffusion‑based RL improve sample efficiency. <br>• Successes in complex games (StarCraft\u202fII, Atari\u202f100k) and real‑world domains (autonomous driving, logistics). | • Safe exploration, reward hacking, transfer from simulation to the real world. |\n| **AI Alignment & Safety** | • Growing work on interpretability, verification, “steerability.” <br>• Institutional safety teams (OpenAI, DeepMind, Partnership on AI) produce standards & red‑team tests. | • Scalable oversight, value learning, preventing emergent deceptive behavior. |\n| **Ethics, Governance & Regulation** | • EU AI Act (2024) and U.S. AI Bill of Rights drafts define risk categories, transparency, provenance requirements. <br>• Rise of bias audits, model‑cards, provenance tracking. | • Harmonising global rules; enforcing accountability for black‑box systems. |\n| **Applications & Industry Adoption** | • Enterprise: generative code assistants, synthetic data, AI‑augmented analytics. <br>• Healthcare: extended protein‑folding, multimodal diagnostics, drug discovery. <br>• Creative media: text‑to‑video, AI‑driven game design, real‑time avatars. | • Integration with legacy IT, data‑security, domain‑specific validation. |\n\n---\n\n### Take‑away Summary\n\n1. **Momentum in Capabilities** – Massive, versatile foundation models and multimodal/embodied AI are the driving force of research.  \n2. **Shift Toward Efficiency** – Because raw scaling is expensive, the community prioritises sparsity, quantisation, and specialised hardware to make models greener and more deployable.  \n3. **Safety & Governance Growing in Urgency** – Alignment, interpretability, and emerging regulatory frameworks (EU AI Act, U.S. AI Bill of Rights) aim to ensure that powerful models are trustworthy and socially beneficial.  \n4. **Broadening Applications, Still Early in Deployment** – AI is moving from prototypes to production in many sectors (enterprise, health, creative media), but robust validation, legacy‑system integration, and responsible deployment remain the main bottlenecks.\n\n**Bottom line:** The AI field stands at a crossroads where ever‑greater technical capability (large, generalist models) meets an increasing emphasis on responsibility (efficiency, safety, and regulation). This tension will shape the next wave of research and real‑world impact.",
}